In [2]:
# Instalação se necessário
# !mamba install biopython pandas numpy tqdm -y
# !pip install biopython pandas numpy tqdm

from Bio import SeqIO
import numpy as np
import pandas as pd
from itertools import product
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- 1. Arquivos FASTA ---
fasta_files = {
    "DENV1": "Dados/DENV1/denv1_NS1_nr.fasta.fasta",
    "DENV2": "Dados/DENV2/denv2_NS1_nr.fasta.fasta",
    "DENV3": "Dados/DENV3/denv3_NS1_nr.fasta.fasta",
    "DENV4": "Dados/DENV4/denv4_NS1_nr.fasta.fasta",
}

# --- 2. Ler sequências ---
groups = {s: [str(r.seq) for r in SeqIO.parse(f, "fasta")] for s, f in fasta_files.items()}

# --- 3. Função de distância genética (p-distance / Hamming) ---
def p_distance(seqA, seqB):
    assert len(seqA) == len(seqB)
    return sum(a != b for a, b in zip(seqA, seqB)) / len(seqA)

# --- 4. Inicializar matriz 4x4 ---
sorotipos = sorted(groups.keys())
matrix = pd.DataFrame(index=sorotipos, columns=sorotipos, dtype=float)

# --- 5. Preencher diagonal (intra-sorotipo) se já tiver, senão calcular ---
# Aqui você pode colocar suas médias intra-sorotipo do FastME
# Exemplo fictício:
matrix.loc["DENV1", "DENV1"] = 0.0801
matrix.loc["DENV2", "DENV2"] = 0.0854
matrix.loc["DENV3", "DENV3"] = 0.0593
matrix.loc["DENV4", "DENV4"] = 0.0735

# --- 6. Função para calcular distância média inter-sorotipo ---
def mean_pairwise_distance(seqs1, seqs2):
    pairwise = [p_distance(a, b) for a, b in product(seqs1, seqs2) if len(a) == len(b)]
    return np.mean(pairwise)

# --- 7. Preparar pares de sorotipos ---
pairs = [(s1, s2) for i, s1 in enumerate(sorotipos) for j, s2 in enumerate(sorotipos) if i < j]

# --- 8. Calcular em paralelo com ThreadPoolExecutor ---
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {}
    for s1, s2 in pairs:
        seqs1 = groups[s1]
        seqs2 = groups[s2]
        futures[executor.submit(mean_pairwise_distance, seqs1, seqs2)] = (s1, s2)

    # Barra de progresso
    for f in tqdm(as_completed(futures), total=len(futures), desc="Calculando inter-sorotipo"):
        s1, s2 = futures[f]
        mean_dist = f.result()
        matrix.loc[s1, s2] = mean_dist
        matrix.loc[s2, s1] = mean_dist  # simétrica

# --- 9. Mostrar e salvar ---
matrix.to_csv("Matriz_Distancias_DENV1-4_paralelo.csv")

Calculando inter-sorotipo: 100%|██████████| 6/6 [03:47<00:00, 37.85s/it]


In [4]:
print(matrix)

          DENV1     DENV2     DENV3     DENV4
DENV1  0.080100  0.299853  0.264333  0.325368
DENV2  0.299853  0.085400  0.309562  0.323778
DENV3  0.264333  0.309562  0.059300  0.316941
DENV4  0.325368  0.323778  0.316941  0.073500
